In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score
from collections import defaultdict

# Oppgave 1

In [2]:
def load_vowel_data(filename):
    vowel_data = {}

    with open(filename, "r") as file:
        for line in file:
            parts = line.split()
            label = parts[0][-2:]  # De to siste tegnene i ID-en er vokalen, feks m01ae så er ae vokalen
            features = list(map(int, parts[10:13]))  # Henter kolonne 11, 12 og 13, hvis vi velger 50%, skjønner ikke helt hva man skal velge?
            
            if label not in vowel_data:
                vowel_data[label] = []
            vowel_data[label].append(features)

    # Konverter til numpy-arrayer og splitt i trening/test
    for vowel in vowel_data:
        vowel_data[vowel] = np.array(vowel_data[vowel])
    
    return vowel_data

# Les inn data fra filen, usikker på om vi skal lage en main som bruker funksjonene tilslutt?
filename = 'vowdata_nohead.dat'
vowel_data = load_vowel_data(filename)

# Splitt inn i trening (første 70) og test (siste 69)
train_data = {vowel: data[:70] for vowel, data in vowel_data.items()}
test_data = {vowel: data[70:] for vowel, data in vowel_data.items()}


In [3]:
# Funksjon for å beregne middelverdi og kovariansmatrise
def compute_mean_covariance(data):
    mean_vectors = {}
    covariance_matrices = {}

    for vowel, samples in data.items():
        mean_vectors[vowel] = np.mean(samples, axis=0)  # finn Middelverdi for hver vokal
        covariance_matrices[vowel] = np.cov(samples, rowvar=False)  # finner kovariansmatrise for hver vokal

    return mean_vectors, covariance_matrices

# Beregn for treningsdataene:
mean_vectors, covariance_matrices = compute_mean_covariance(train_data)

#printer bare for å se egt, kan fjernes før innlevering
for vowel in mean_vectors:
    print(f"\nVokal: {vowel}")
    print(f"Middelverdi: {mean_vectors[vowel]}")
    print(f"Kovariansmatrise:\n{covariance_matrices[vowel]}")

print



Vokal: ae
Middelverdi: [ 661.64285714 1956.71428571 2687.22857143]
Kovariansmatrise:
[[ 6602.05900621  7416.99792961 10198.63354037]
 [ 7416.99792961 31858.75776398 32295.18219462]
 [10198.63354037 32295.18219462 54018.58467909]]

Vokal: ah
Middelverdi: [ 815.88571429 1410.12857143 2627.35714286]
Kovariansmatrise:
[[11988.4505176  11334.10186335 13338.91097308]
 [11334.10186335 27903.96873706 13177.40269151]
 [13338.91097308 13177.40269151 42944.46480331]]

Vokal: aw
Middelverdi: [ 718.6        1115.84285714 2608.8       ]
Kovariansmatrise:
[[ 7085.86666667  8096.60289855 10026.16521739]
 [ 8096.60289855 17766.4242236  12058.40289855]
 [10026.16521739 12058.40289855 46864.0173913 ]]

Vokal: eh
Middelverdi: [ 640.15714286 1869.54285714 2735.17142857]
Kovariansmatrise:
[[ 6500.22132505  9124.63809524 13937.05962733]
 [ 9124.63809524 31255.7300207  30458.54327122]
 [13937.05962733 30458.54327122 52099.65134576]]

Vokal: ei
Middelverdi: [ 454.87142857 2332.88571429 2581.11428571]
Kovarian

<function print>

In [4]:
# Funksjon for å klassifisere en testprøve med Gaussisk sannsynlighetstetthet
def classify_sample(x, mean_vectors, cov_matrices):
    best_class = None #vi vet ikke hva denne er enda
    max_prob = -np.inf #setter lavest mulig tall så vi finner den høyeste
    
    for vowel, mean in mean_vectors.items():
        cov = cov_matrices[vowel] 
        det_cov = np.linalg.det(cov)
      
        cov_inv = np.linalg.inv(cov)
        
        diff = x - mean
        exponent = -0.5 * np.dot(diff.T, np.dot(cov_inv, diff))
        prob = np.exp(exponent) / np.sqrt((2 * np.pi) ** 3 * det_cov) #Bruker PDF

        if prob > max_prob:
            max_prob = prob
            best_class = vowel

    return best_class


In [5]:
true_labels = []
predicted_labels = []

for vowel, samples in test_data.items():
    for sample in samples:
        predicted_vowel = classify_sample(sample, mean_vectors, covariance_matrices)
        true_labels.append(vowel)
        predicted_labels.append(predicted_vowel)
        
# Beregn forvirringsmatrise og feilrate
conf_matrix = confusion_matrix(true_labels, predicted_labels, labels=list(vowel_data.keys()))
error_rate = 1 - accuracy_score(true_labels, predicted_labels)


print("Forvirringsmatrise (full kovarians):")
print(conf_matrix)
print(f"Feilrate: {error_rate*100:.2f}%")


Forvirringsmatrise (full kovarians):
[[52  1  0 10  1  2  3  0  0  0  0  0]
 [ 1 61  2  2  0  0  0  0  0  0  3  0]
 [ 0 32 29  0  0  1  0  0  0  0  6  1]
 [28  0  0 37  0  0  4  0  0  0  0  0]
 [ 0  0  0  0 41  0  5 23  0  0  0  0]
 [33  0  0  1  1 34  0  0  0  0  0  0]
 [ 1  0  0  0 21  0 44  3  0  0  0  0]
 [ 0  0  0  0 16  0  0 53  0  0  0  0]
 [ 0  0  2  0  0  0  0  0 53  3  5  6]
 [ 0  0  0  8  0  6  0  0  1 49  4  1]
 [ 0  3  2  9  0  0  0  0  0  7 47  1]
 [ 0  0  0  2  5 13  1  0 18  5  0 25]]
Feilrate: 36.59%


In [6]:
# Lager diagonalmatrisene
diag_cov_matrices = {}

for vowel in mean_vectors:
    identity_matrix = np.identity(3)
    diag_cov_matrices[vowel] = covariance_matrices[vowel]*identity_matrix
    print(f'Diagonal matrix for {vowel}:\n {diag_cov_matrices[vowel]}\n ')



Diagonal matrix for ae:
 [[ 6602.05900621     0.             0.        ]
 [    0.         31858.75776398     0.        ]
 [    0.             0.         54018.58467909]]
 
Diagonal matrix for ah:
 [[11988.4505176      0.             0.        ]
 [    0.         27903.96873706     0.        ]
 [    0.             0.         42944.46480331]]
 
Diagonal matrix for aw:
 [[ 7085.86666667     0.             0.        ]
 [    0.         17766.4242236      0.        ]
 [    0.             0.         46864.0173913 ]]
 
Diagonal matrix for eh:
 [[ 6500.22132505     0.             0.        ]
 [    0.         31255.7300207      0.        ]
 [    0.             0.         52099.65134576]]
 
Diagonal matrix for ei:
 [[  1775.24409938      0.              0.        ]
 [     0.          61156.18964803     -0.        ]
 [     0.             -0.         670950.30559006]]
 
Diagonal matrix for er:
 [[  1834.75465839      0.              0.        ]
 [     0.          17360.49772257      0.        ]
 [  

In [7]:
# Klassifiser testsettet på nytt med diagonal kovarians
predicted_labels_diag = []

for vowel, samples in test_data.items():
    for sample in samples:
        predicted_vowel = classify_sample(sample, mean_vectors, diag_cov_matrices)
        predicted_labels_diag.append(predicted_vowel)

# Beregn forvirringsmatrise og feilrate for diagonal kovarians
conf_matrix_diag = confusion_matrix(true_labels, predicted_labels_diag, labels=list(vowel_data.keys()))
error_rate_diag = 1 - accuracy_score(true_labels, predicted_labels_diag)

print("Forvirringsmatrise (diagonal kovarians):")
print(conf_matrix_diag)
print(f"Feilrate: {error_rate_diag*100:.2f}%")


Forvirringsmatrise (diagonal kovarians):
[[62  1  0  0  5  0  0  1  0  0  0  0]
 [ 6 63  0  0  0  0  0  0  0  0  0  0]
 [ 0 46 13  0  0  1  0  0  0  0  9  0]
 [65  0  0  4  0  0  0  0  0  0  0  0]
 [ 2  0  0  0 55  0  4  8  0  0  0  0]
 [16  0  0 12  0 38  2  0  0  0  1  0]
 [ 5  0  0  0 42  0 19  3  0  0  0  0]
 [ 0  0  0  0 21  0  0 48  0  0  0  0]
 [ 0  0 10  0  0  0  0  0 28 13 15  3]
 [ 0  0  0 30  0  0  1  0  2 17 19  0]
 [ 1 17  2 29  0  0  0  0  0  0 19  1]
 [ 0  0  0  3  0  5  7  0 18 27  1  8]]
Feilrate: 54.83%


Dårligere, gir mening siden vi kun bruker diagonalen av covariansmartrisen, som er det samme som å anta uavhengighet mellom featurene. Det er vel litt naivt...

Er det kanskje litt vel dårlig?


# Oppgave 2

In [8]:
from sklearn.mixture import GaussianMixture



bruker bare trening og testdata jeg har fra hør,
har train_data og test_data

In [9]:

"""
gmm_i = GaussianMixture(n_components=M, covariance_type=’full’, random_state=42) #will put M mixtures into GMMi using the training vectors trainvi from class ωi.
gmm_i.fit(trainv_i) 
likelihoods = gmm_i.score_samples(test_set)"""

# Funksjon for å trene GMM-modeller, uansett antall modeller vil vel denne fungere
def train_gmm_models(train_data, n_components):
    gmm_models = {}
    
    for vowel, samples in train_data.items():
        gmm = GaussianMixture(n_components=n_components, covariance_type='diag', random_state=42)
        gmm.fit(samples)
        gmm_models[vowel] = gmm
    
    return gmm_models 

In [10]:
# Funksjon for å klassifisere en prøve med GMM-modeller, oppgradering fra 1b
def classify_sample_gmm(sample, gmm_models):
    max_log_likelihood = -np.inf #setter uendelig lavt for å kunne oppdatere til høyeste
    best_class = None #vil finne, setter til none foreløpig
    
    for vowel, gmm in gmm_models.items():
        log_likelihood = gmm.score_samples(sample.reshape(1, -1))[0] #skjønner ikke helt shapinga
        
        if log_likelihood > max_log_likelihood:
            max_log_likelihood = log_likelihood
            best_class = vowel
    
    return best_class

In [11]:
#oppdatert evaluering til å passse ulik gmm
def evaluate_classifier(test_data, gmm_models):
    true_labels = []
    predicted_labels = []
    
    for vowel, samples in test_data.items():
        for sample in samples:
            predicted_vowel = classify_sample_gmm(sample, gmm_models)
            true_labels.append(vowel)
            predicted_labels.append(predicted_vowel)
    
    # Beregn confusionmatrise og feilrate
    conf_matrix = confusion_matrix(true_labels, predicted_labels, labels=list(test_data.keys()))
    error_rate = 1 - accuracy_score(true_labels, predicted_labels)
    
    return conf_matrix, error_rate


In [13]:
# bruker alle funksjonene

# Tren GMM-modeller med 2 og 3 komponenter
gmm_models_2 = train_gmm_models(train_data, n_components=2)
gmm_models_3 = train_gmm_models(train_data, n_components=3)

# Evaluere modellene, 2 og 2 gmm
conf_matrix_2, error_rate_2 = evaluate_classifier(test_data, gmm_models_2)
conf_matrix_3, error_rate_3 = evaluate_classifier(test_data, gmm_models_3)

# print:) Fiks på format senere
print("Forvirringsmatrise (GMM, 2 komponenter):")
print(conf_matrix_2)
print(f"Feilrate: {error_rate_2*100:.2f}")

print("\nForvirringsmatrise (GMM, 3 komponenter):")
print(conf_matrix_3)
print(f"Feilrate: {error_rate_3*100:.2f}")


Forvirringsmatrise (GMM, 2 komponenter):
[[55  1  0  7  0  0  1  5  0  0  0  0]
 [ 4 60  2  0  0  0  0  0  0  0  3  0]
 [ 0 30 31  0  0  1  0  0  0  0  7  0]
 [42  0  0 24  0  0  3  0  0  0  0  0]
 [ 0  0  0  0 24  0  5 40  0  0  0  0]
 [21  0  0  4  0 44  0  0  0  0  0  0]
 [ 2  0  0  0  7  0 40 20  0  0  0  0]
 [ 0  0  0  0  8  0  0 61  0  0  0  0]
 [ 0  0  1  0  0  0  0  0 46  9  6  7]
 [ 0  0  0 14  0  0  1  0  1 44  8  1]
 [ 1  8  1 11  0  0  0  0  0  2 45  1]
 [ 0  0  0  2  6  1  1  0 20 16  0 23]]
Feilrate: 39.98

Forvirringsmatrise (GMM, 3 komponenter):
[[46  0  0 17  1  0  1  4  0  0  0  0]
 [ 2 59  3  2  0  0  0  0  0  0  3  0]
 [ 0 31 28  0  0  1  0  0  0  0  9  0]
 [43  0  0 23  0  0  3  0  0  0  0  0]
 [ 0  0  0  0 34  0  2 33  0  0  0  0]
 [14  0  0  3  0 52  0  0  0  0  0  0]
 [ 2  0  0  0 12  0 40 15  0  0  0  0]
 [ 0  0  0  0 10  0  0 59  0  0  0  0]
 [ 0  0  1  0  0  0  0  0 45  8  6  9]
 [ 0  0  0 12  0  0  0  0  1 47  7  2]
 [ 0  5  1 13  0  0  0  0  0  2 47  1]
 [ 

**(d) Compare the performances for all four model types (tasks 1b, 1c and 2c). For
which class(es) is the difference largest?**

- Ser at det er best performance for GMM 3 blandt de med kun diagonal, men ikke så stor forskjell fra GMM 2
- Vanlig full covarians er best, kan se for full covarian på de to siste om det blir bedre?
- GMM2 er best på de første, gmm 3 er best på de siste
- Må vel sammenligne mer nøye

In [16]:
#Setter full covariansmatriser

def train_gmm_models_full(train_data, n_components):
    gmm_models = {}
    
    for vowel, samples in train_data.items():
        gmm = GaussianMixture(n_components=n_components, covariance_type='full', random_state=42)
        gmm.fit(samples)
        gmm_models[vowel] = gmm
    
    return gmm_models 

# Tren GMM-modeller med 2 og 3 komponenter
gmm_models_2 = train_gmm_models_full(train_data, n_components=2)
gmm_models_3 = train_gmm_models_full(train_data, n_components=3)

# Evaluere modellene, 2 og 2 gmm
conf_matrix_2, error_rate_2 = evaluate_classifier(test_data, gmm_models_2)
conf_matrix_3, error_rate_3 = evaluate_classifier(test_data, gmm_models_3)


print("Forvirringsmatrise (GMM med fullcovarians, 2 komponenter):")
print(conf_matrix_2)
print(f"Feilrate: {error_rate_2*100:.2f}")

print("\nForvirringsmatrise (GMM med full kovarians, 3 komponenter):")
print(conf_matrix_3)
print(f"Feilrate: {error_rate_3*100:.2f}")

Forvirringsmatrise (GMM med fullcovarians, 2 komponenter):
[[51  1  0 12  1  3  1  0  0  0  0  0]
 [ 3 59  1  3  0  0  0  0  0  0  3  0]
 [ 0 26 31  0  0  1  0  0  0  0 11  0]
 [34  0  0 32  0  0  3  0  0  0  0  0]
 [ 0  0  0  0 37  0  3 29  0  0  0  0]
 [13  1  0  1  0 54  0  0  0  0  0  0]
 [ 5  0  0  0 10  0 49  5  0  0  0  0]
 [ 0  0  0  0  7  0  1 61  0  0  0  0]
 [ 0  0  2  0  0  0  0  0 48 11  3  5]
 [ 0  0  0  6  0  0  0  0  1 53  6  3]
 [ 0  5  2 11  0  0  0  0  0  4 47  0]
 [ 0  0  0  2  0  1  2  0 22 12  0 30]]
Feilrate: 33.33

Forvirringsmatrise (GMM med full kovarians, 3 komponenter):
[[38  1  0 24  2  3  1  0  0  0  0  0]
 [ 3 57  1  3  0  0  0  0  0  0  5  0]
 [ 0 27 27  0  0  1  0  0  0  1 11  2]
 [29  0  0 37  0  0  3  0  0  0  0  0]
 [ 0  0  0  0 35  0  4 30  0  0  0  0]
 [13  0  0  1  0 55  0  0  0  0  0  0]
 [ 1  0  0  0  8  0 56  4  0  0  0  0]
 [ 0  0  0  0  7  0  1 61  0  0  0  0]
 [ 0  0  1  0  0  0  0  0 47 11  3  7]
 [ 1  0  0  5  0  0  0  0  1 53  8  1]
 [ 0 

Definitivt best, men usikker på om dette er del av oppgaven... Men her er plutselig 2 bedre enn 3